# Constellation Detection — Colab Submission

This notebook uses only the course-provided images, reference patterns, and training labels in `constellation_colab_bundle.zip`. Do not add external data, labels, or pretrained models.

The standard path produces a strictly validated Kaggle CSV. A valid file is not a guarantee of any leaderboard score.

## Runtime

A normal Colab CPU runtime is sufficient. The joint matcher and geometric solver are CPU-based; a GPU is optional.

In [ ]:
# Choose one source for the bundle.
USE_GOOGLE_DRIVE = False
DRIVE_ARCHIVE = '/content/drive/MyDrive/constellation_colab_bundle.zip'

from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    archive = Path(DRIVE_ARCHIVE)
else:
    from google.colab import files
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload exactly one file: constellation_colab_bundle.zip')
    archive = archives[0]

if not archive.is_file():
    raise FileNotFoundError(archive)
print(f'Using bundle: {archive} ({archive.stat().st_size / 1024**2:.1f} MiB)')

In [ ]:
# Extract into Colab's local disk. Re-run safely to reset the project files.
import shutil
import zipfile

ROOT = Path('/content/constellation-detection')
shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True)
with zipfile.ZipFile(archive) as bundle:
    bundle.extractall(ROOT)
print(ROOT)
print(sorted(item.name for item in ROOT.iterdir()))

In [ ]:
# Install the image-processing packages used by this project. PyTorch is preinstalled on GPU Colab runtimes.
%pip install -q --upgrade scipy pillow opencv-python-headless

In [ ]:
# Verify the supplied dataset before spending runtime.
import csv
import cv2
import numpy as np

required = ['train', 'validation', 'patterns', 'sample_submission.csv', 'train_ground_truth.csv',
            'constellation_pipeline.py', 'structural_refiner.py', 'joint_geometric_solver.py',
            'matcher_config_fast.json']
missing = [name for name in required if not (ROOT / name).exists()]
if missing:
    raise FileNotFoundError(f'Incomplete bundle; missing: {missing}')

with (ROOT / 'sample_submission.csv').open(newline='', encoding='utf-8') as stream:
    template_rows = list(csv.DictReader(stream))
assert len(template_rows) == 16, f'Expected 16 sample rows, found {len(template_rows)}'
assert len(list((ROOT / 'patterns').glob('*_pattern.png'))) == 48
print('Dataset check passed:', len(template_rows), 'submission rows and 48 supplied patterns')
print('The default joint solver is CPU-based and runs on a normal Colab runtime.')

In [ ]:
# Calibrate the present/absent threshold using only train_ground_truth.csv.
import subprocess

OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)
def run(*args):
    print('+', ' '.join(map(str, args)))
    subprocess.run(list(map(str, args)), cwd=ROOT, check=True)

CONFIG = OUT / 'matcher_config_colab.json'
run('python', 'constellation_pipeline.py', 'calibrate', '--root', ROOT,
    '--config', ROOT / 'matcher_config_fast.json', '--output', CONFIG)

In [ ]:
# Final method: retain each patch's top 16 locations, then use the supplied 48
# constellation graphs to choose a geometrically consistent one-to-one subset.
# Candidate results are cached, so re-running this cell after an interruption
# resumes without repeating completed per-scene patch matching.
FINAL_CSV = OUT / 'submission_final_colab.csv'
CACHE = OUT / 'joint_candidate_cache'
run('python', 'joint_geometric_solver.py', '--root', ROOT, '--config', CONFIG,
    '--output', FINAL_CSV, '--top-k', '16', '--proposals', '6000', '--cache-dir', CACHE)
run('python', 'constellation_pipeline.py', 'validate', '--root', ROOT, '--output', FINAL_CSV)

In [ ]:
# The output above is the final candidate. The next cell performs independent
# CSV checks before the download is triggered.

In [ ]:
# Inspect the file exactly as it will be uploaded. No null or blank fields are permitted.
with FINAL_CSV.open(newline='', encoding='utf-8') as stream:
    reader = csv.reader(stream)
    header = next(reader)
    submission_rows = list(reader)
assert (len(submission_rows), len(header)) == (16, 90), (len(submission_rows), len(header))
assert all(len(row) == len(header) for row in submission_rows), 'Inconsistent CSV row width'
assert not any(cell == '' for row in submission_rows for cell in row), 'Found an empty CSV cell'
print(header[:8])
print(submission_rows[0][:8])
print('Ready for Kaggle:', FINAL_CSV, (len(submission_rows), len(header)))

In [ ]:
# Download this exact file and upload it manually on Kaggle.
from google.colab import files
files.download(str(FINAL_CSV))

## Optional experimental geometric run

Leave this disabled unless you have time for a separate experiment. It is course-data-only, but it is not validated as an improvement over the normal CSV. Never overwrite the normal CSV before recording its Kaggle score.

In [ ]:
RUN_EXPERIMENTAL_GEOMETRY = False
GEOMETRIC_CSV = OUT / 'submission_geometric_colab.csv'

if RUN_EXPERIMENTAL_GEOMETRY:
    run('python', 'geometric_ensemble.py', '--root', ROOT, '--output', GEOMETRIC_CSV,
        '--per-mode', '12', '--proposals', '20000')
    run('python', 'constellation_pipeline.py', 'validate', '--root', ROOT, '--output', GEOMETRIC_CSV)
    print('Experimental CSV:', GEOMETRIC_CSV)